![Henry Logo](https://www.soyhenry.com/_next/static/media/HenryLogo.bb57fd6f.svg)

# M3L2 E10 - RAG mini: el pipeline completo (Resolution)

## BLOQUE 1 — ¿Qué es RAG y por qué existe?

**RAG** = Retrieval Augmented Generation (Generación Aumentada por Recuperación).

### El problema

Un LLM entrenado hasta cierta fecha **no sabe** información específica de tu empresa, documentos privados o datos actualizados. Además, puede **alucinar** (inventar) respuestas cuando no sabe algo.

Soluciones posibles:

| Enfoque | Qué hace | Problema |
|---|---|---|
| **Fine-tuning** | Re-entrenar el modelo con tus datos | Caro, lento, hay que repetirlo si los datos cambian |
| **Mandar todo el contexto** | Poner todos los documentos en el prompt | Carísimo en tokens, no escala a miles de documentos |
| **RAG** | Recuperar SOLO los documentos relevantes y pasarlos al LLM | Escala, es dinámico, no requiere re-entrenar |

### El flujo RAG

```text
FASE DE INDEXACION (se hace 1 vez):
  Documentos -> Splitter -> Chunks -> Embeddings -> Vector Store

FASE DE CONSULTA (se hace por cada pregunta):
  Pregunta -> Embedding -> Buscar en Vector Store -> Documentos relevantes
                                                        |
                                                        v
  Prompt (contexto + pregunta) -> LLM -> Respuesta
```

### ¿Por qué RAG es mejor que el script legacy?

| Aspecto | Script legacy (manda TODO) | RAG (solo lo relevante) |
|---|---|---|
| **Costo por consulta** | Alto (todos los documentos como tokens) | Bajo (solo k documentos) |
| **Calidad** | El LLM recibe ruido de documentos irrelevantes | Contexto limpio y enfocado |
| **Escala** | No escala: 10 docs = 10x tokens | Escala: 10.000 docs = mismo k=3 |
| **Actualización** | Hay que modificar el código si los docs cambian | Solo cambiar los textos en el vector store |
| **Debugging**| Difícil saber qué información usó el LLM | Podés inspeccionar los docs recuperados |

## BLOQUE 2 — Setup

In [ ]:
import os, getpass
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Ingresa tu OpenAI API key: ")
print("API key cargada.")

## BLOQUE 3 — Script legacy: el enfoque que NO queremos

Este script **manda todos los documentos** como contexto en cada consulta. Con 5 documentos no duele, pero con 5000 documentos es inviable.

Problemas específicos:
- **Cada llamada** embebe 0 documentos (no hay retrieval real)
- El prompt contiene **todo** el texto, incluso documentos que no tienen nada que ver con la pregunta
- Si los documentos cambian, hay que editar el código
- No hay forma de saber qué documentos usó el modelo para responder

In [ ]:
from openai import OpenAI

client = OpenAI()

DOCS_EMPRESA = [
    "La politica de vacaciones es de 15 dias por ano.",
    "Los empleados tienen seguro medico incluido desde el primer dia.",
    "El horario de trabajo es de 9 a 18 con 1 hora de almuerzo.",
    "El trabajo remoto esta habilitado 3 dias por semana previa aprobacion del manager.",
    "Los bonos anuales se calculan en base al desempeno y se pagan en diciembre.",
]


def answer_script_legacy(question: str) -> str:
    context = "\n".join(DOCS_EMPRESA)  # TODO: siempre
    prompt_text = (
        "Eres un asistente de RRHH. Responde usando solo el contexto dado.\n\n"
        f"Contexto:\n{context}\n\nPregunta:\n{question}"
    )
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt_text}],
        temperature=0,
    )
    return response.choices[0].message.content


print("Script legacy:")
print(answer_script_legacy("Cuantos dias de vacaciones tengo?"))
print()
print(f"Tokens aproximados del contexto: {sum(len(d) for d in DOCS_EMPRESA)} chars")

## BLOQUE 4 — Pipeline RAG modular con LangChain

El pipeline RAG tiene estos componentes, cada uno reemplazable independientemente:

| Componente | Rol | Se crea con... |
|---|---|---|
| **Embeddings** | Convierte texto en vectores numéricos | `OpenAIEmbeddings()` |
| **Vector Store** | Almacena vectores y permite búsqueda por similitud | `FAISS.from_texts()` |
| **Retriever** | Interfaz estándar para recuperar documentos | `vectorstore.as_retriever()` |
| **PromptTemplate** | Arma el prompt con contexto + pregunta | `ChatPromptTemplate.from_messages()` |
| **LLM** | Genera la respuesta basada en el contexto | `ChatOpenAI()` |
| **Parser** | Extrae el texto del AIMessage | `StrOutputParser()` |
| **LCEL** | Conecta todo con el operador `|` | `dict \| prompt \| llm \| parser` |

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

TEXTOS = [
    "La politica de vacaciones es de 15 dias por ano.",
    "Los empleados tienen seguro medico incluido desde el primer dia.",
    "El horario de trabajo es de 9 a 18 con 1 hora de almuerzo.",
    "El trabajo remoto esta habilitado 3 dias por semana previa aprobacion del manager.",
    "Los bonos anuales se calculan en base al desempeno y se pagan en diciembre.",
]

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings()
parser = StrOutputParser()

print(f"LLM: {llm.model_name}")
print(f"Embeddings: {embeddings.model}")

### TODO 1: Crear el Vector Store

`FAISS.from_texts(TEXTOS, embeddings)` hace 3 cosas:
1. Embede cada texto
2. Indexa los vectores en FAISS
3. Guarda los textos originales asociados a cada vector

In [ ]:
# TODO 1
vectorstore = FAISS.from_texts(TEXTOS, embeddings)
print(f"Vector store: {type(vectorstore).__name__}")
print(f"Documentos indexados: {vectorstore.index.ntotal}")

### TODO 2: Crear el Retriever

`vectorstore.as_retriever(search_kwargs={"k": 2})` envuelve el vector store en una interfaz estándar.

**k=2** significa que solo vamos a recuperar los 2 documentos más relevantes para cada pregunta, en vez de mandar los 5 (como hace el script legacy).

In [ ]:
# TODO 2
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
print(f"Retriever: {type(retriever).__name__}, k=2")

# Probar que el retriever funciona
docs_test = retriever.invoke("vacaciones")
print(f"Documentos recuperados para 'vacaciones': {len(docs_test)}")
for i, doc in enumerate(docs_test):
    print(f"  Doc {i+1}: {doc.page_content}")

### TODO 3: Crear el Prompt RAG

El prompt tiene 2 variables:
- `{context}`: los documentos recuperados (solo los relevantes, no todos)
- `{question}`: la pregunta del usuario

El mensaje `system` le indica al modelo que solo use el contexto dado. Esto **previene alucinaciones**.

In [ ]:
# TODO 3
rag_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Eres un asistente de RRHH. "
        "Responde usando solo el contexto proporcionado. "
        "Si la respuesta no esta en el contexto, indica que no tienes informacion suficiente."
    ),
    (
        "human",
        "Contexto:\n{context}\n\nPregunta:\n{question}"
    )
])
print(f"Prompt RAG creado. Variables: {rag_prompt.input_variables}")

### TODO 4: Componer la RAG chain con LCEL

Esta es la parte más interesante. La chain usa **RunnablePassthrough** para pasar la pregunta directamente al prompt, mientras que el contexto se obtiene del retriever.

```python
rag_chain = (
    {
        "context": retriever | format_docs,   # recupera y formatea docs
        "question": RunnablePassthrough()       # pasa la pregunta tal cual
    }
    | rag_prompt                                  # arma el mensaje
    | llm                                         # genera respuesta
    | parser                                      # extrae el texto
)
```

El flujo completo:

```text
"Cuantos dias de vacaciones tengo?"
  |
  v (RunnablePassthrough -> pasa la pregunta)
  |
  v (retriever | format_docs -> busca y formatea)
  |
  +---> dict {"context": "La politica de vacaciones...", "question": "Cuantos dias...?"}
          |
          v (rag_prompt)
          Mensaje system + human con contexto y pregunta
          |
          v (llm)
          AIMessage
          |
          v (parser)
          "Segun la politica, tienes 15 dias de vacaciones por ano."
```

In [ ]:
def format_docs(docs) -> str:
    return "\n\n".join(doc.page_content for doc in docs)


# TODO 4
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | llm
    | parser
)

print(f"RAG chain compuesta: {type(rag_chain).__name__}")
print()
print("La chain completa: retriever -> format -> prompt -> llm -> parser")

### TODO 5: Invocar la RAG chain

Ahora la pregunta pasa por todo el pipeline: retrieval → formato → prompt → LLM → parser.

In [ ]:
# TODO 5
respuesta = rag_chain.invoke("Cuantos dias de vacaciones tengo?")
print(f"Respuesta RAG: {respuesta}")
print()
print("Con RAG, el LLM solo recibe los 2 documentos mas relevantes (no los 5).")

## BLOQUE 5 — Debugging modular

Una de las grandes ventajas del pipeline modular es que **cada paso es inspeccionable por separado**. Si la respuesta es mala, sabemos exactamente dónde buscar.

In [ ]:
question = "Cuantos dias de vacaciones tengo?"

print("========== DEBUG DEL PIPELINE ==========")

print("\n[Paso 1] Documentos recuperados por el retriever:")
docs = retriever.invoke(question)
for i, doc in enumerate(docs):
    print(f"  Doc {i+1}: {doc.page_content}")

print("\n[Paso 2] Contexto formateado (entrada al prompt):")
context = format_docs(docs)
print(f"  {context}")

print("\n[Paso 3] Mensajes del prompt (entrada al LLM):")
messages = rag_prompt.format_messages(context=context, question=question)
for msg in messages:
    print(f"  [{msg.type.upper()}] {msg.content[:100]}...")

print("\n[Paso 4] Respuesta final:")
print(f"  {rag_chain.invoke(question)}")
print()
print("=========================================")

## BLOQUE 6 — Errores comunes

| Error | Consecuencia | Solución |
|---|---|---|
| No usar `format_docs` | El prompt recibe `List[Document]` en vez de string | Siempre convertir los docs a texto con `\n\n`.join() |
| `k` muy bajo (k=1) | El contexto puede no tener la respuesta | Empezá con k=3 y ajustá |
| `k` muy alto (k=10) | Demasiados tokens, ruido en el contexto | No más de k=5 para documentos pequeños |
| No usar `system` message | El modelo no sabe que debe limitarse al contexto | Incluí "Responde solo con el contexto dado" |
| Olvidar `RunnablePassthrough` | Error porque falta la variable `question` | Usá `RunnablePassthrough()` para pasar el input original |
| No testear el retriever por separado | No sabés si el problema es retrieval o generación | Probá `retriever.invoke(query)` antes de armar la chain |
| Usar embeddings distintos en indexación y consulta | El retriever no encuentra docs relevantes | Usá SIEMPRE el mismo objeto `embeddings` |

## BLOQUE 7 — Comparación final: script legacy vs RAG

| Aspecto | Script legacy | Pipeline RAG con LangChain |
|---|---|---|
| **Retrieval** | Manda todos los docs siempre | Solo los k más relevantes |
| **Costo de tokens** | 5 docs por consulta = ~500 chars | 2 docs por consulta = ~200 chars |
| **Precisión** | El LLM recibe ruido de docs irrelevantes | Contexto limpio y enfocado |
| **Prompt** | F-string hardcodeado dentro de la función | `ChatPromptTemplate` reutilizable e inspeccionable |
| **Modelo** | Hardcodeado en `client.chat.completions.create()` | Objeto `ChatOpenAI` reemplazable |
| **Debugging** | Opaco (hay que leer la función completa) | Cada paso se puede inspeccionar por separado |
| **Escalabilidad** | No escala (10 docs = 10x tokens) | Escala (10.000 docs = mismo k=3) |
| **Mantenimiento** | Cambiar documentos = editar código | Cambiar documentos = actualizar vector store |

### ¿Cuándo usar RAG?

- Cuando tenés **documentos propios** que el LLM no conoce
- Cuando necesitás **información actualizada** (no dependiente del training cutoff)
- Cuando querés **evitar alucinaciones** forzando contexto
- Cuando tenés **muchos documentos** y no podés ponerlos todos en el prompt

### ¿Cuándo NO usar RAG?

- Para tareas de conocimiento general (el LLM ya lo sabe)
- Para respuestas que no requieren contexto externo
- Para prototipos de 1 consulta sin documentos

## BLOQUE 8 — Checks automáticos

In [ ]:
def run_checks():
    assert vectorstore is not None
    assert retriever is not None
    assert rag_prompt is not None
    assert rag_chain is not None

    docs = retriever.invoke("vacaciones")
    assert len(docs) > 0
    assert len(docs) <= 2
    contenidos = [doc.page_content for doc in docs]
    assert any("vacaciones" in c.lower() or "15 dias" in c.lower() for c in contenidos)

    respuesta = rag_chain.invoke("Cuantos dias de vacaciones tengo?")
    assert isinstance(respuesta, str)
    assert len(respuesta) > 0
    assert "15" in respuesta

    docs_remoto = retriever.invoke("trabajo remoto")
    contenidos_remoto = [doc.page_content for doc in docs_remoto]
    assert any("remoto" in c.lower() for c in contenidos_remoto)

    print("M3L2 E10 Resolution checks passed")

run_checks()

## Cierre

Hoy aprendiste:

1. **RAG** = Recuperar solo los documentos relevantes y pasarlos al LLM como contexto
2. El **script legacy** manda todos los documentos siempre — no escala, es caro y propenso a ruido
3. El **pipeline RAG** con LangChain es modular: cada componente es reemplazable e inspeccionable
4. **`RunnablePassthrough()`** permite pasar la pregunta directamente al prompt mientras el retriever trabaja
5. **Debugging modular**: podés inspeccionar retriever, contexto, prompt y respuesta por separado
6. **Comparación**: RAG gana en costo, precisión, escalabilidad y mantenibilidad

### Próximos pasos

- **E11**: RAG chat con memoria (historial + retrieval combinados)
- **E13**: refactorizar un script caótico usando todo lo aprendido